# 06_DL_Models.ipynb
## FakeJobShield: Deep Learning Text Models
This notebook designs, trains, and evaluates three Keras recurrent neural networks (LSTM, Bidirectional LSTM, and GRU) on job text data. Early stopping and checkpoints are employed to prevent overfitting and save the best weights.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import joblib
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D, Bidirectional, GRU
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score

# Adjust working directory if run from notebooks folder
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sns.set_theme(style="whitegrid")


In [ ]:
# Load dataset
df = pd.read_csv("data/cleaned_fake_job_postings.csv")
df["cleaned_text"] = df["cleaned_text"].fillna("")
df["fraudulent_int"] = df["fraudulent"].map({'t': 1, 'f': 0, '1': 1, '0': 0, 1: 1, 0: 0}).fillna(0).astype(int)

texts = df["cleaned_text"].values
labels = df["fraudulent_int"].values


In [ ]:
# Text Tokenization & Padding
MAX_WORDS = 10000
MAX_LEN = 200

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
padded_seqs = pad_sequences(sequences, maxlen=MAX_LEN, padding="post", truncating="post")

print("Padded shape:", padded_seqs.shape)


In [ ]:
# Split Data
x_train, x_test, y_train, y_test = train_test_split(
    padded_seqs, labels, test_size=0.20, stratify=labels, random_state=42
)
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.10, stratify=y_train, random_state=42
)

print(f"Train size: {x_train.shape[0]} | Val size: {x_val.shape[0]} | Test size: {x_test.shape[0]}")


In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)


In [ ]:
# 1. Simple LSTM Model
lstm_model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN),
    SpatialDropout1D(0.2),
    LSTM(64, dropout=0.2, recurrent_dropout=0.0),
    Dense(32, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

lstm_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
lstm_model.summary()

print("Training LSTM Model...")
lstm_hist = lstm_model.fit(
    x_train, y_train,
    epochs=3,
    batch_size=128,
    validation_data=(x_val, y_val),
    callbacks=[early_stop, ModelCheckpoint("models/lstm_best.keras", save_best_only=True)],
    verbose=1
)


In [ ]:
# 2. Bidirectional LSTM Model
bilstm_model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.0)),
    Dense(32, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

bilstm_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
bilstm_model.summary()

print("Training Bidirectional LSTM Model...")
bilstm_hist = bilstm_model.fit(
    x_train, y_train,
    epochs=3,
    batch_size=128,
    validation_data=(x_val, y_val),
    callbacks=[early_stop, ModelCheckpoint("models/bilstm_best.keras", save_best_only=True)],
    verbose=1
)


In [ ]:
# 3. GRU Model
gru_model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN),
    SpatialDropout1D(0.2),
    GRU(64, dropout=0.2, recurrent_dropout=0.0),
    Dense(32, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

gru_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
gru_model.summary()

print("Training GRU Model...")
gru_hist = gru_model.fit(
    x_train, y_train,
    epochs=3,
    batch_size=128,
    validation_data=(x_val, y_val),
    callbacks=[early_stop, ModelCheckpoint("models/gru_best.keras", save_best_only=True)],
    verbose=1
)


In [ ]:
# Plot curves
def plot_histories(histories, names):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    for hist, name in zip(histories, names):
        ax1.plot(hist.history["val_accuracy"], label=f"{name} Val Acc")
        ax2.plot(hist.history["val_loss"], label=f"{name} Val Loss")
        
    ax1.set_title("Validation Accuracy")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Accuracy")
    ax1.legend()
    
    ax2.set_title("Validation Loss")
    ax2.set_xlabel("Epochs")
    ax2.set_ylabel("Loss")
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig("results/dl_val_curves.png", dpi=150)
    plt.show()

plot_histories([lstm_hist, bilstm_hist, gru_hist], ["LSTM", "BiLSTM", "GRU"])


In [ ]:
# Evaluate DL models
dl_results = {}
for model, name in [(lstm_model, "LSTM"), (bilstm_model, "BiLSTM"), (gru_model, "GRU")]:
    probs = model.predict(x_test).flatten()
    preds = (probs >= 0.5).astype(int)
    
    f1 = f1_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    
    dl_results[name] = {
        "F1 Score": f1,
        "ROC AUC": auc
    }
    print(f"{name} Test Performance:")
    print(f"  F1: {f1:.4f} | ROC AUC: {auc:.4f}")

# Save DL comparison
with open("results/dl_model_metrics.json", "w") as f:
    json.dump(dl_results, f, indent=2)

# Save tokenizer and best model
joblib.dump(tokenizer, "models/dl_tokenizer.pkl")
bilstm_model.save("models/best_dl_model.keras")
print("Saved tokenizer and finished DL model benchmark.")
